# 常见问题定位方法

模型接入昇腾后，可能遇到编译失败、执行结果异常或性能不达预期等问题。本节介绍 GE 中常见的排障方法，包括日志、错误码、图 dump 和运行时问题定位。

本节学习大纲如下：

- 故障处理总体流程
- 日志级别与错误码体系（ASCEND_GLOBAL_LOG_LEVEL / 错误码命名规则 / GEGetErrorMsg）
- 图 dump 分析（DUMP_GE_GRAPH / DUMP_GRAPH_LEVEL 各级别产物与可视化）
- 编译失败典型案例 + checklist（算子不支持 / InferShape 失败 / 属性校验失败）
- 运行时问题定位（模型加载 / 输入输出不匹配 / 内存与 stream 同步）
- 数据 dump 与精度定位
- 常用定位工具速查（asys / msaicerr / msprof）
- 排障总流程图

> 本节命令、错误码、工具能力均取自 GE / 故障处理文档；屏显报错以 `[%s]` 形式代表实际变量，实际以你的运行环境为准。

## 1. 故障处理总体流程

昇腾故障处理总体分三步：收集故障信息 → 分析故障原因 → 故障排除。

<p align="left"><img src="./images/troubleshooting_flow.svg" alt="故障处理三步法" width="75%"></p>

故障信息主要分三类：

| 类别 | 内容 | 获取方式 |
| --- | --- | --- |
| 应用类日志 | Host/Device 用户态日志，最常用的是 **plog**（`plog-<pid>_*.log`） | 直接查看日志目录 |
| Device 侧系统类日志/维测信息 | 内核态日志、系统进程日志等 | 需用 **msnpureport** 导出到 Host |
| dump / 编译信息 | GE dump 图、算子编译 `.o`/`.json`、异常现场 | 环境变量开关 / asys 收集 |

> 分析时遵循**自上而下的日志分析法**：从应用层报错入手，沿业务流程逐步缩小到底层故障现象。异步执行场景可能有多个算子报错，应从首报错算子开始排查。

## 2. 日志级别与错误码体系

### 2.1 日志级别配置

通过环境变量控制 CANN 日志级别（全局生效），排障时通常将日志级别的数值调低，以输出更详细的日志：

```bash
# 全局日志级别：0-DEBUG 1-INFO 2-WARNING 3-ERROR 4-NULL(关闭)
export ASCEND_GLOBAL_LOG_LEVEL=1          # 排障常用 INFO/DEBUG

# 日志输出位置：0 落盘文件 / 1 输出到 stdout
export ASCEND_SLOG_PRINT_TO_STDOUT=0      # 0 落盘文件 / 1 打屏

# Event 日志开关：0 关闭 / 1 开启
export ASCEND_GLOBAL_EVENT_ENABLE=0

# 模块级日志（示例；模块名和格式以当前版本文档为准）
# export ASCEND_MODULE_LOG_LEVEL=GE=0:RUNTIME=1
```

| 级别值 | 含义 | 使用建议 |
| --- | --- | --- |
| 0 | DEBUG | 最详细，深度定位时用 |
| 1 | INFO | 常规排障，能看到关键流程 |
| 2 | WARNING | 默认偏静默 |
| 3 | ERROR | 仅报错 |

最常用的应用日志是 **plog**，路径形如 `log/[run|debug]/plog/plog-<pid>_*.log`。排障时常按时间窗 + 关键字检索：

```bash
# 在 plog 中查首个 ERROR 及上下文
grep -n "ERROR" plog-12345_*.log | head
# 关注关键字：内存 mem_stats / 算子 not supported / shape mismatch / ECC 等
```

### 2.2 错误码命名规则

错误码为 6 位字符，例如 `E10035`：

```
E  1  0035
│  │   └── 后 4 位：0000~8999 用户类错误；9000~9999 内部错误码
│  └────── 第 2 位：模块（完整模块码见下表）
└───────── 第 1 位：级别（E=错误, W=告警, I=提示）
```

| 模块码（完整列表） | 模块 | 模块码 | 模块 |
| --- | --- | --- | --- |
| 1 | GE | E | Runtime |
| 2 | FE（融合引擎） | B | TBE Pass 编译工具 |
| 3 | AI CPU | C | Auto Tune |
| 4 | TE Fusion | D | RLTune |
| 7 | Vector 算子插件 | F | LxFusion & AutoDeploy |
| 8 | Vector 算子 | G | AOE |
| H | ACL | I | HCCL |
| J | HCCP | K | Profiling |
| L | Driver | M | 队列调度 |
| N | DVPP | O | AMCT |
| P | Dump | Z | 算子公共 / AclNN API |

> 看到 `E19999` / 形如 `E*9***` 的错误码，多为**系统内部错误**，需联系华为技术支持。用户类错误（0000~8999）一般可按手册「Possible Cause / Solution」自助处理。

### 2.3 程序内获取错误信息

GE 提供接口在执行报错时获取错误描述（头文件 `ge/ge_api.h`，库 `libge_runner.so`）：

```cpp
#include "ge/ge_api.h"
// 某 GE 接口返回失败时，调用获取错误描述（获取并清空当前进程/线程的错误信息）
std::string msg = ge::GEGetErrorMsg();   // 推荐使用 GEGetErrorMsgV2
// 打印 msg 辅助定位；注意与 GEGetErrorMsgV2 不要同时使用
```

## 3. 图 dump 分析

当编译报错或结果异常时，**导出计算图**是最直接的「看清 GE 到底构了/优化成什么样」的手段。GE 通过环境变量自动 dump 图。

### 3.1 dump 开关

```bash
# 内容粒度：1=全量(含连边+数据) 2=基本版(不含权重数据) 3=精简版(只显示节点关系)
export DUMP_GE_GRAPH=2

# dump 输出路径（绝对或相对，支持中文）
export DUMP_GRAPH_PATH=/path/to/dump

# dump 格式：ge_proto | onnx | readable，多个用 | 分隔（默认 ge_proto|onnx）
export DUMP_GRAPH_FORMAT="ge_proto|onnx|readable"

# 控制 dump 哪些编译阶段的图
export DUMP_GRAPH_LEVEL=2
```

### 3.2 DUMP_GRAPH_LEVEL 各级别

| 取值 | dump 内容 | 适用 |
| --- | --- | --- |
| 1 | 所有阶段的图 | 需要看完整编译演化过程 |
| 2（默认） | 白名单阶段的图 | 常规排障，产物适中 |
| 3 | 最后的生成图（经 GE 优化、编译后） | 看最终下沉图长什么样 |
| 4 | 最早的生成图（解析映射算子后的编译入口图） | 看「刚进 GE」的原始结构 |
| 字符串 | 如 `"PreRunBegin\|AfterInfershape"` | dump 名称包含这些字符串的图（精准定位某阶段） |

> 排查 **InferShape 问题**时，对比 `AfterInfershape` 前后的图特别有用；排查**算子映射**问题时，看 level=4 的编译入口图。

### 3.3 三种 dump 格式

| 格式 | 文件名 | 特点 | 用途 |
| --- | --- | --- | --- |
| ge_proto | `ge_proto*.txt` | protobuf 文本，**信息最完整**，可转 JSON | 精确定位、看全量属性 |
| onnx | `ge_onnx*.pbtxt` | ONNX 风格，**可视化** | 用 Netron 打开看结构/连边 |
| readable | `ge_readable*.txt` | 类 Dynamo fx 风格，**可读性最高** | 快速通读图逻辑 |

readable 格式示例（节点以「函数调用」方式展示 type/inputs/attrs）：

```
graph("MakeSubGraph"):
  %input_0 : [#users=1] = Node[type=Data] (attrs = {index: 0})
  %Const_0 : [#users=1] = Node[type=Const] (attrs = {value: [-1 7168]})
  %Reshape_1 : [#users=1] = Node[type=Reshape] (inputs = (x=%input_0, shape=%Const_0), attrs = {axis: 0})
  %MatMul_6 : [#users=1] = Node[type=MatMul] (inputs = (x1=%Cast_2, x2=%Transpose_5), attrs = {transpose_x1: false})
```

### 3.4 Netron 可视化要点

打开 `ge_onnx*.pbtxt` 后，点击算子节点可看：`type`（算子类型）、`name`（算子名）、`input_desc_shape:x` / `input_desc_dtype:x` / `input_desc_layout:x`（第 x 个输入的 shape/dtype/format），输出同理。连边用带箭头实线表示数据流向。

> 排障第一步常常就是：dump 图 → 找到报错算子 → 看它的输入输出 shape/dtype/format 是否符合预期。

## 4. 编译失败典型案例 + checklist

编译期（图准备/拆分/优化/编译）失败，最常见的是下面三类。

### 4.1 算子不支持 / 未注册

| 子类 | 关键日志 | 含义 |
| --- | --- | --- |
| 算子插件未注册 | `Check op[%s]'s type[%s] failed, it is not supported.` 或被转成 `frameworkop` | TF/ONNX→GE 的映射插件 so 未加载或缺映射 |
| 算子原型未注册 | `IR for op[%s] optype[%s] is not registered.` / `have no ir factory` | 算子原型 so 未加载或原型未编译进 so |

定位 checklist：

```
[ ] 1. 查日志是否有 "Plugin load .../framework/<framework>/<plugin>.so success."（框架适配插件；实际检索建议使用 grep -i）
[ ] 2. 查日志是否有 "OpsProtoManager plugin load .../op_proto/<prototype>.so successfully."（原型）
[ ] 3. 失败则看 "dlopen failed, plugin name:%s. Message(%s)." 的 Message
[ ] 4. 确认 ASCEND_OPP_PATH 等环境变量是否正确配置
[ ] 5. nm -D xxx.so | grep <算子类型>  确认是否注册进 so
[ ] 6. 仍缺失则按算子开发指南补充注册（插件/原型）
```

> 对应错误码示例：`E10501 Not_Supported_Operator`（IR 未注册），可能原因含「ASCEND_OPP_PATH 未配置」「IR 未注册」。

### 4.2 InferShape 失败

现象：图准备阶段推导输出 shape 失败，或推导结果与预期不符。常见根因：

- 输入 shape/dtype/format 与算子约束不匹配；
- 动态 shape 下 shape 范围设置不当（如 `INPUT_SHAPE` 的范围与真实输入冲突）；
- 上游算子输出 shape 异常，沿数据流传播放大。

checklist：

```
[ ] 1. dump 图（level 含 AfterInfershape），对比 InferShape 前后报错算子的 shape
[ ] 2. 检查该算子各输入的 shape/dtype/format 是否满足算子原型约束
[ ] 3. 动态 shape：核对 INPUT_SHAPE 范围 / 档位是否覆盖真实输入
[ ] 4. 顺数据流向上找「第一个 shape 不对」的算子，往往是真正源头
```

### 4.3 属性（Attr）校验失败

现象：算子属性非法 / 缺失，常见错误码如 `E14002 Invalid_Argument_Tensor_Attribute`、`E10002 ...Tensor_Input_Shape` 等。

checklist：

```
[ ] 1. 看报错指明的算子 name/type 与属性名
[ ] 2. 对照算子原型，确认属性取值范围 / 是否必填
[ ] 3. 构图/Parser 配置处修正属性（如 axis、dtype、perm 等）
[ ] 4. dump readable 图，确认该算子 attrs 实际值
```

> 通用原则：先看错误码 → 查手册 Possible Cause/Solution → dump 图核对报错算子的输入/属性 → 修正后重编。

## 5. 运行时问题定位

编译通过、执行阶段出问题，按现象分三类。

### 5.1 模型加载失败

| 现象 | 可能原因 | 定位/处理 |
| --- | --- | --- |
| soc 不匹配 | OM 是为其他芯片编译的（soc_version 不一致） | 在目标芯片上用对应 `--soc_version` 重新编译 OM |
| 内存不足 | 加载时申请 Device 内存失败 | 见 5.3；减小 batch / 释放占用 / 排查泄漏 |
| 版本不兼容 | OM 与当前 CANN/Runtime 版本不配套 | 用配套版本重新编译，并核对 driver/runtime 版本 |
| 驱动版本容量不足 | RTS 错误码 `EE1015 Package_Error_Incorrect_Driver_Version` | 升级驱动软件；该错误码不等同于 OM 重新编译问题 |

### 5.2 输入输出 dataset / shape / dtype 不匹配

这是推理部署最高频的运行时报错。典型日志：

```
[ERROR] ... Check][Size] add2(Add) index[0] mem size out of range!
                 Expected size: 128, but given input size: 2.
[ERROR] ... CheckParam.Outputs] Output Size mismatch. index = 0,
                 model expect ..., but given ....
```

含义：用户分配的输入/输出 buffer 大小，与模型 InferShape 推导出的大小不一致（动态 shape 场景尤其常见）。

处理 checklist：

```
[ ] 1. 按报错提示的 Expected size 重新分配 input/output buffer
[ ] 2. 核对喂入数据的 shape/dtype 与模型输入定义是否一致
[ ] 3. 动态 shape：确认实际输入落在编译的 shape 范围/档位内
[ ] 4. 多输入：逐个 index 核对（报错会指明 index）
```

> 对应案例：「动态 shape 模型用户输入和模型推导结果不匹配」「动态 shape 模型输入大小校验失败」。原则是**以模型推导/报错提示的 size 为准来分配 buffer**。



### 5.2.1 动手实践：从首报错到图规格，定位 buffer 大小不匹配

下面使用一个最小示例，演示如何根据运行时日志和图规格定位 buffer 大小不匹配问题：

```text
首条 ERROR → 提取 op/index/Expected size → 在 ge_proto 中核对 shape/dtype → 计算正确 buffer 大小
```

默认夹具模拟 `add2(Add)` 的第 0 个输入：日志显示只给了 4 bytes，而 `ge_proto` 显示输入规格为 `[2, 3] + DT_FLOAT`，因此应分配 `2 × 3 × 4 = 24 bytes`。这一步只使用 Host 文本和 Python 标准库，不需要 NPU。`GE_PLOG_PATH` 与 `GE_GRAPH_DUMP_PATH` 是本教程代码自定义的夹具入口，并非 CANN 官方环境变量；替换真实故障时，同时设置它们即可。示例中的 `math.prod(shape) × dtype_bytes` 只适用于静态、连续的 ND 教学数据，不能泛化到动态 shape、带 padding/stride 的特殊 format 等场景；真实推理应以模型 descriptor 或 `aclmdlGetInputSizeByIndex` / `aclmdlGetOutputSizeByIndex` 返回的大小为准。

In [ ]:
# === 可运行：联合 plog 与 ge_proto 定位输入/输出 buffer 大小不匹配 ===
import math
import os
import re
import tempfile
from pathlib import Path

# 可选：换成真实故障产物。以下两个变量是本教程自定义的夹具入口，
# 不是 CANN 官方环境变量；两项必须同时设置，图文件应为 ge_proto 文本格式。
# os.environ["GE_PLOG_PATH"] = "/path/to/plog-12345.log"
# os.environ["GE_GRAPH_DUMP_PATH"] = "/path/to/ge_proto_xxx.txt"
plog_override = os.environ.get("GE_PLOG_PATH")
graph_override = os.environ.get("GE_GRAPH_DUMP_PATH")
if bool(plog_override) != bool(graph_override):
    raise RuntimeError("GE_PLOG_PATH 与 GE_GRAPH_DUMP_PATH 必须同时设置。")

using_fixture = not plog_override
if using_fixture:
    # 教学夹具模拟真实产物：运行时只给 add2 的 input[0] 分配了 4 bytes，
    # 而图中该输入为 [2, 3] 的 DT_FLOAT，实际需要 2 * 3 * 4 = 24 bytes。
    artifact_dir = Path(tempfile.mkdtemp(prefix="ge_troubleshooting_demo_"))
    plog_path = artifact_dir / "plog-demo.log"
    graph_path = artifact_dir / "ge_proto-demo.txt"
    plog_path.write_text(
        """[INFO] GE model load success, graph_id=0
[ERROR] GE [Check][Size] add2(Add) index[0] mem size out of range!
        Expected size: 24, but given input size: 4.
[ERROR] GE RunGraph failed, ret=145000
""",
        encoding="utf-8",
    )
    graph_path.write_text(
        """graph {
  name: "buffer_mismatch_demo"
  op {
    name: "input_0"
    type: "Data"
    output_desc {
      dtype: DT_FLOAT
      shape { dim: 2 dim: 3 }
    }
  }
  op {
    name: "const_0"
    type: "Const"
    output_desc {
      dtype: DT_FLOAT
      shape { dim: 2 dim: 3 }
    }
  }
  op {
    name: "add2"
    type: "Add"
    input: "input_0:0"
    input: "const_0:0"
    input_desc {
      dtype: DT_FLOAT
      shape { dim: 2 dim: 3 }
    }
    input_desc {
      dtype: DT_FLOAT
      shape { dim: 2 dim: 3 }
    }
    output_desc {
      dtype: DT_FLOAT
      shape { dim: 2 dim: 3 }
    }
  }
}
""",
        encoding="utf-8",
    )
else:
    plog_path = Path(plog_override).expanduser().resolve()
    graph_path = Path(graph_override).expanduser().resolve()
    for path in (plog_path, graph_path):
        if not path.is_file():
            raise FileNotFoundError("故障产物不存在：{}".format(path))

plog_text = plog_path.read_text(encoding="utf-8", errors="replace")
graph_text = graph_path.read_text(encoding="utf-8", errors="replace")


def first_error_context(text, following_lines=3):
    """返回首条 ERROR 及其后若干行，保留跨行的 Expected/given 信息。"""
    lines = text.splitlines()
    for index, line in enumerate(lines):
        if "ERROR" in line.upper():
            return "\n".join(lines[index : index + following_lines + 1])
    raise RuntimeError("plog 中没有找到 ERROR。")


def classify_error(context):
    lowered = context.lower()
    if "expected size" in lowered and "given" in lowered:
        return "INPUT_OUTPUT_BUFFER_SIZE_MISMATCH"
    if "not supported" in lowered or "have no ir factory" in lowered:
        return "OP_NOT_SUPPORTED_OR_NOT_REGISTERED"
    if "infershape" in lowered:
        return "INFER_SHAPE_FAILED"
    if "stream_synchronize_timeout" in lowered or "ee1002" in lowered:
        return "STREAM_SYNCHRONIZE_TIMEOUT"
    if "out of memory" in lowered or "malloc" in lowered:
        return "MEMORY_ALLOCATION_FAILED"
    return "UNCLASSIFIED"


def iter_braced_blocks(text, keyword):
    """从 protobuf 文本中提取形如 `keyword { ... }` 的平衡括号块。"""
    pattern = re.compile(r"\b{}\s*\{{".format(re.escape(keyword)))
    for match in pattern.finditer(text):
        open_brace = text.find("{", match.start())
        depth = 0
        for position in range(open_brace, len(text)):
            if text[position] == "{":
                depth += 1
            elif text[position] == "}":
                depth -= 1
                if depth == 0:
                    yield text[match.start() : position + 1]
                    break


def find_op_block(text, op_name):
    for block in iter_braced_blocks(text, "op"):
        name_match = re.search(r'\bname\s*:\s*"([^"]+)"', block)
        if name_match and name_match.group(1) == op_name:
            return block
    return None


error_context = first_error_context(plog_text)
category = classify_error(error_context)
size_pattern = re.compile(
    r"(?P<op>[A-Za-z0-9_./-]+)\((?P<op_type>[^)]+)\).*?"
    r"index\[(?P<index>\d+)\].*?Expected size:\s*(?P<expected>\d+).*?"
    r"given(?:\s+(?P<io_kind>input|output))?\s+size:\s*(?P<given>\d+)",
    re.IGNORECASE | re.DOTALL,
)
size_match = size_pattern.search(error_context)

print("产物来源：", "内置教学夹具" if using_fixture else "真实 plog + ge_proto")
print("首报错上下文：\n{}".format(error_context))
print("故障分类：", category)

if category == "INPUT_OUTPUT_BUFFER_SIZE_MISMATCH" and size_match:
    details = size_match.groupdict()
    op_name = details["op"]
    io_kind = (details.get("io_kind") or "input").lower()
    tensor_index = int(details["index"])
    expected_bytes = int(details["expected"])
    given_bytes = int(details["given"])
    print(
        "日志定位：op={}({}), {}_index={}, expected={} bytes, given={} bytes".format(
            op_name, details["op_type"], io_kind, tensor_index, expected_bytes, given_bytes
        )
    )

    op_block = find_op_block(graph_text, op_name)
    if op_block is None:
        print("[WARN] ge_proto 中未找到算子 {}，请确认日志与 dump 来自同一次运行。".format(op_name))
    else:
        desc_key = "input_desc" if io_kind == "input" else "output_desc"
        descs = list(iter_braced_blocks(op_block, desc_key))
        if tensor_index >= len(descs):
            print("[WARN] 算子 {} 没有 {}[{}]。".format(op_name, desc_key, tensor_index))
        else:
            desc = descs[tensor_index]
            dtype_match = re.search(r"\bdtype\s*:\s*(DT_[A-Z0-9_]+)", desc)
            format_match = re.search(r"\bformat\s*:\s*(FORMAT_[A-Z0-9_]+)", desc)
            shape_blocks = list(iter_braced_blocks(desc, "shape"))
            dims = re.findall(r"\bdim\s*:\s*(-?\d+)", shape_blocks[0] if shape_blocks else "")
            dims = [int(value) for value in dims]
            dtype = dtype_match.group(1) if dtype_match else "UNKNOWN"
            data_format = format_match.group(1) if format_match else None
            dtype_bytes = {
                "DT_BOOL": 1,
                "DT_INT8": 1,
                "DT_UINT8": 1,
                "DT_FLOAT16": 2,
                "DT_BF16": 2,
                "DT_INT16": 2,
                "DT_UINT16": 2,
                "DT_FLOAT": 4,
                "DT_INT32": 4,
                "DT_UINT32": 4,
                "DT_DOUBLE": 8,
                "DT_INT64": 8,
                "DT_UINT64": 8,
            }.get(dtype)
            if data_format not in (None, "FORMAT_ND"):
                print("[WARN] format={} 可能包含 padding/stride，不能用逻辑 shape 直接计算 storage bytes；请以 ACL/model descriptor 为准。".format(data_format))
            elif not dims or any(dim < 0 for dim in dims) or dtype_bytes is None:
                print("[WARN] 无法静态计算字节数：shape={}, dtype={}".format(dims, dtype))
            else:
                # 仅对静态 FORMAT_ND 教学样例计算逻辑字节数；真实模型以 ACL descriptor size 为准。
                graph_bytes = math.prod(dims) * dtype_bytes
                print(
                    "图中规格：{}[{}] shape={}, dtype={} ({} bytes/element) -> {} bytes".format(
                        io_kind, tensor_index, dims, dtype, dtype_bytes, graph_bytes
                    )
                )
                if graph_bytes != expected_bytes:
                    raise AssertionError(
                        "日志 Expected size={}，但图规格计算为 {}。请确认 dump 与日志匹配。".format(
                            expected_bytes, graph_bytes
                        )
                    )
                corrected_host_buffer = bytearray(expected_bytes)
                assert len(corrected_host_buffer) == expected_bytes
                print("修复建议：buffer 至少增加 {} bytes，并重新核对 shape/dtype。".format(expected_bytes - given_bytes))
                print("Host 演示分配：bytearray({}) -> {} bytes".format(expected_bytes, len(corrected_host_buffer)))
                print("[OK] 首报错、算子规格与 Expected size 三方一致")
else:
    print("未命中 buffer 大小不匹配规则；请按故障分类进入本节对应 checklist。")

print("plog：", plog_path)
print("ge_proto：", graph_path)

### 5.2.2 可选实战：ACL descriptor 与真实 NPU buffer 校验

上一小节用 Host 侧夹具演示了「日志 → 图规格 → Expected size」的分析过程。本 cell 进一步连接真实 ACL 运行时：先故意把第 0 个输入的 DataBuffer 元数据标记为过小，观察 aclmdlExecute 返回失败；然后按 aclmdlDesc 查询到的实际 size 重新准备 dataset 并执行成功。

底层 Device 内存仍按完整 descriptor size 申请，因此故意失败的尝试不会造成越界写。示例按本章 CANN 9.0.0 基线的 ACL C API 编写，并通过 `aclrtGetRunMode` 自动兼容 `ACL_HOST` 与 `ACL_DEVICE` 数据路径；要求使用本节内置 Add 图生成的 OM（可由 05.03 阶段二生成），并需要 CANN toolkit 的 ACL 头文件/库和真实 NPU。cell 内会固定启用设备执行，直接运行即可。

~~~bash
# 使用 05.03 阶段二生成的 OM，也可以改成 02.02 的 add_sample.om
export GE_TROUBLESHOOTING_OM=/tmp/ge_tf_parser_demo/tf_add.om
# GE_TROUBLESHOOTING_RUN_NPU=1 已由 cell 自动设置，无需手动 export
~~~

若未设置 `GE_TROUBLESHOOTING_OM`，代码会自动查找 05.03 生成的 `/tmp/ge_tf_parser_demo/tf_add.om`；仍找不到 OM 时才会在编译 runner 后明确跳过。若找到 OM，cell 会直接启动真实 NPU 执行。`GE_TROUBLESHOOTING_OM` 与 `GE_TROUBLESHOOTING_RUN_NPU` 是本教程的控制变量，并非 CANN 官方环境变量。

In [ ]:
# === 可选实战：ACL descriptor 与真实 NPU buffer 校验 ===
import os
import shutil
import subprocess
import textwrap
from pathlib import Path

# 本 cell 是真实 NPU 排障实战，固定启用；直接运行即可。
os.environ["GE_TROUBLESHOOTING_RUN_NPU"] = "1"
print("GE_TROUBLESHOOTING_RUN_NPU =", os.environ["GE_TROUBLESHOOTING_RUN_NPU"])


def run_acl_buffer_troubleshooting():
    ascend_home_value = os.environ.get("ASCEND_HOME_PATH")
    if not ascend_home_value:
        print("[SKIP] 未设置 ASCEND_HOME_PATH，跳过 ACL runner 编译。")
        return
    if shutil.which("g++") is None:
        print("[SKIP] 未找到 g++，跳过 ACL runner 编译。")
        return

    ascend_home = Path(ascend_home_value).expanduser()
    prefix_candidates = [ascend_home, ascend_home / "x86_64-linux"]
    prefix = next(
        (
            candidate
            for candidate in prefix_candidates
            if (candidate / "include/acl/acl.h").is_file()
            and (candidate / "lib64/libacl_mdl.so").is_file()
            and (candidate / "lib64/libacl_rt.so").is_file()
        ),
        None,
    )
    if prefix is None:
        print("[SKIP] 找不到 ACL 头文件或 libacl_mdl.so/libacl_rt.so。")
        return

    library_dirs = [prefix / "lib64"]
    root_libdir = ascend_home / "lib64"
    if root_libdir.is_dir() and root_libdir not in library_dirs:
        library_dirs.append(root_libdir)
    rpath = os.pathsep.join(str(path) for path in library_dirs)

    configured_om = os.environ.get("GE_TROUBLESHOOTING_OM")
    if configured_om:
        om_path = Path(configured_om).expanduser().resolve()
    else:
        candidates = [
            Path("/tmp/ge_tf_parser_demo/tf_add.om"),
            Path("add_sample.om").resolve(),
            Path("tf_add.om").resolve(),
        ]
        om_path = next((candidate for candidate in candidates if candidate.is_file()), None)

    workdir = Path("/tmp/ge_acl_troubleshooting_demo")
    workdir.mkdir(parents=True, exist_ok=True)

    runner_source = textwrap.dedent(
        r'''
        #include <cmath>
        #include <cstdint>
        #include <cstring>
        #include <iostream>
        #include <vector>

        #include "acl/acl.h"
        #include "acl/acl_mdl.h"
        #include "acl/acl_rt.h"

        namespace {
        struct DatasetBuffers {
          aclmdlDataset *dataset = nullptr;
          std::vector<void *> ptrs;
          std::vector<size_t> allocation_sizes;
        };

        void DestroyDataset(DatasetBuffers *buffers) {
          if (buffers == nullptr || buffers->dataset == nullptr) {
            return;
          }
          const size_t count = aclmdlGetDatasetNumBuffers(buffers->dataset);
          for (size_t i = 0; i < count; ++i) {
            aclDataBuffer *buffer = aclmdlGetDatasetBuffer(buffers->dataset, i);
            if (buffer != nullptr) {
              (void)aclDestroyDataBuffer(buffer);
            }
          }
          for (void *ptr : buffers->ptrs) {
            if (ptr != nullptr) {
              (void)aclrtFree(ptr);
            }
          }
          (void)aclmdlDestroyDataset(buffers->dataset);
          buffers->dataset = nullptr;
          buffers->ptrs.clear();
          buffers->allocation_sizes.clear();
        }

        bool CreateDataset(aclmdlDesc *desc, bool input,
                           size_t advertised_input0_bytes,
                           DatasetBuffers *buffers) {
          buffers->dataset = aclmdlCreateDataset();
          if (buffers->dataset == nullptr) {
            return false;
          }
          const size_t count = input ? aclmdlGetNumInputs(desc)
                                     : aclmdlGetNumOutputs(desc);
          for (size_t i = 0; i < count; ++i) {
            const size_t descriptor_bytes =
                input ? aclmdlGetInputSizeByIndex(desc, i)
                      : aclmdlGetOutputSizeByIndex(desc, i);
            const size_t advertised_bytes =
                (input && i == 0 && advertised_input0_bytes != 0)
                    ? advertised_input0_bytes
                    : descriptor_bytes;

            // 即使故意把 DataBuffer 元数据设小，底层分配仍使用 descriptor_bytes，
            // 保证错误尝试不会越界写 Device 内存。
            void *ptr = nullptr;
            if (aclrtMalloc(&ptr, descriptor_bytes,
                            ACL_MEM_MALLOC_NORMAL_ONLY) != ACL_SUCCESS) {
              return false;
            }
            aclDataBuffer *buffer = aclCreateDataBuffer(ptr, advertised_bytes);
            if (buffer == nullptr ||
                aclmdlAddDatasetBuffer(buffers->dataset, buffer) != ACL_SUCCESS) {
              if (buffer != nullptr) {
                (void)aclDestroyDataBuffer(buffer);
              }
              (void)aclrtFree(ptr);
              return false;
            }
            buffers->ptrs.push_back(ptr);
            buffers->allocation_sizes.push_back(descriptor_bytes);
          }
          return true;
        }

        bool CopyAddInputs(const DatasetBuffers &inputs,
                           aclrtRunMode run_mode) {
          const std::vector<std::vector<float>> host_inputs = {
              {1.0F, 2.0F, 3.0F, 4.0F, 5.0F, 6.0F},
              {10.0F, 20.0F, 30.0F, 40.0F, 50.0F, 60.0F},
          };
          if (inputs.ptrs.size() != host_inputs.size()) {
            return false;
          }
          for (size_t i = 0; i < host_inputs.size(); ++i) {
            const size_t bytes = host_inputs[i].size() * sizeof(float);
            if (inputs.allocation_sizes[i] < bytes ||
                aclrtMemset(inputs.ptrs[i], inputs.allocation_sizes[i], 0,
                            inputs.allocation_sizes[i]) != ACL_SUCCESS) {
              return false;
            }
            if (run_mode == ACL_HOST) {
              if (aclrtMemcpy(inputs.ptrs[i], inputs.allocation_sizes[i],
                              host_inputs[i].data(), bytes,
                              ACL_MEMCPY_HOST_TO_DEVICE) != ACL_SUCCESS) {
                return false;
              }
            } else if (run_mode == ACL_DEVICE) {
              std::memcpy(inputs.ptrs[i], host_inputs[i].data(), bytes);
            } else {
              std::cerr << "unsupported aclrtRunMode="
                        << static_cast<int>(run_mode) << "\n";
              return false;
            }
          }
          return true;
        }
        }  // namespace

        int main(int argc, char **argv) {
          if (argc != 2) {
            std::cerr << "usage: acl_buffer_troubleshooting MODEL.om\n";
            return 2;
          }
          if (aclInit(nullptr) != ACL_SUCCESS) {
            std::cerr << "aclInit failed\n";
            return 1;
          }

          uint32_t device_count = 0;
          if (aclrtGetDeviceCount(&device_count) != ACL_SUCCESS ||
              device_count == 0) {
            std::cerr << "no available Ascend NPU device\n";
            (void)aclFinalize();
            return 3;
          }
          if (aclrtSetDevice(0) != ACL_SUCCESS) {
            std::cerr << "aclrtSetDevice(0) failed\n";
            (void)aclFinalize();
            return 1;
          }
          aclrtRunMode run_mode = ACL_HOST;
          if (aclrtGetRunMode(&run_mode) != ACL_SUCCESS) {
            std::cerr << "aclrtGetRunMode failed\n";
            (void)aclrtResetDevice(0);
            (void)aclFinalize();
            return 1;
          }
          std::cout << "run_mode="
                    << (run_mode == ACL_HOST ? "ACL_HOST" : "ACL_DEVICE")
                    << "\n";

          uint32_t model_id = 0;
          if (aclmdlLoadFromFile(argv[1], &model_id) != ACL_SUCCESS) {
            std::cerr << "aclmdlLoadFromFile failed\n";
            (void)aclrtResetDevice(0);
            (void)aclFinalize();
            return 1;
          }
          aclmdlDesc *desc = aclmdlCreateDesc();
          auto cleanup = [&]() {
            (void)aclmdlDestroyDesc(desc);
            (void)aclmdlUnload(model_id);
            (void)aclrtResetDevice(0);
            (void)aclFinalize();
          };

          if (desc == nullptr || aclmdlGetDesc(desc, model_id) != ACL_SUCCESS) {
            std::cerr << "aclmdlGetDesc failed\n";
            if (desc != nullptr) {
              (void)aclmdlDestroyDesc(desc);
            }
            (void)aclmdlUnload(model_id);
            (void)aclrtResetDevice(0);
            (void)aclFinalize();
            return 1;
          }
          if (aclmdlGetNumInputs(desc) != 2 ||
              aclmdlGetNumOutputs(desc) != 1) {
            std::cerr << "this demo expects the built-in float32 Add OM with\n"
                      << "two 24-byte inputs and one 24-byte output\n";
            cleanup();
            return 1;
          }

          const size_t input0_bytes = aclmdlGetInputSizeByIndex(desc, 0);
          const size_t input1_bytes = aclmdlGetInputSizeByIndex(desc, 1);
          const size_t output_bytes = aclmdlGetOutputSizeByIndex(desc, 0);
          std::cout << "descriptor_input0_bytes=" << input0_bytes << "\n";
          std::cout << "descriptor_input1_bytes=" << input1_bytes << "\n";
          std::cout << "descriptor_output0_bytes=" << output_bytes << "\n";
          const size_t expected_bytes = 6 * sizeof(float);
          if (input0_bytes < expected_bytes ||
              input1_bytes < expected_bytes ||
              output_bytes < expected_bytes) {
            std::cerr << "descriptor is smaller than the six-element float32 demo\n";
            cleanup();
            return 1;
          }

          if (input0_bytes > sizeof(float)) {
            DatasetBuffers wrong_inputs;
            DatasetBuffers wrong_outputs;
            const size_t wrong_advertised_bytes = sizeof(float);
            if (!CreateDataset(desc, true, wrong_advertised_bytes, &wrong_inputs) ||
                !CreateDataset(desc, false, 0, &wrong_outputs) ||
                !CopyAddInputs(wrong_inputs, run_mode)) {
              DestroyDataset(&wrong_inputs);
              DestroyDataset(&wrong_outputs);
              cleanup();
              return 1;
            }
            const aclError wrong_status =
                aclmdlExecute(model_id, wrong_inputs.dataset, wrong_outputs.dataset);
            std::cout << "intentional_advertised_input0_bytes="
                      << wrong_advertised_bytes << "\n";
            std::cout << "intentional_aclmdlExecute_status="
                      << static_cast<int>(wrong_status) << "\n";
            if (wrong_status == ACL_SUCCESS) {
              std::cout << "[WARN] 当前运行时接受了故意缩小的 DataBuffer 元数据；"
                           "继续用 descriptor size 做正确执行。\n";
            } else {
              std::cout << "[OK] ACL 按预期拒绝了过小的输入 buffer。\n";
            }
            DestroyDataset(&wrong_inputs);
            DestroyDataset(&wrong_outputs);
          } else {
            std::cout << "[SKIP] 输入 descriptor 小于等于一个 float，"
                         "无法构造有意义的过小 buffer 对照。\n";
          }

          DatasetBuffers inputs;
          DatasetBuffers outputs;
          if (!CreateDataset(desc, true, 0, &inputs) ||
              !CreateDataset(desc, false, 0, &outputs) ||
              !CopyAddInputs(inputs, run_mode)) {
            DestroyDataset(&inputs);
            DestroyDataset(&outputs);
            cleanup();
            return 1;
          }
          if (aclmdlExecute(model_id, inputs.dataset, outputs.dataset) != ACL_SUCCESS) {
            std::cerr << "correct aclmdlExecute failed\n";
            DestroyDataset(&inputs);
            DestroyDataset(&outputs);
            cleanup();
            return 1;
          }

          const std::vector<float> expected = {
              11.0F, 22.0F, 33.0F, 44.0F, 55.0F, 66.0F};
          if (output_bytes < expected.size() * sizeof(float) ||
              output_bytes % sizeof(float) != 0) {
            std::cerr << "unexpected output descriptor size\n";
            DestroyDataset(&inputs);
            DestroyDataset(&outputs);
            cleanup();
            return 1;
          }
          std::vector<float> actual(output_bytes / sizeof(float));
          if (run_mode == ACL_HOST) {
            if (aclrtMemcpy(actual.data(), output_bytes, outputs.ptrs[0],
                            output_bytes,
                            ACL_MEMCPY_DEVICE_TO_HOST) != ACL_SUCCESS) {
              std::cerr << "aclrtMemcpy(output D2H) failed\n";
              DestroyDataset(&inputs);
              DestroyDataset(&outputs);
              cleanup();
              return 1;
            }
          } else {
            std::memcpy(actual.data(), outputs.ptrs[0], output_bytes);
          }

          float max_error = 0.0F;
          bool finite = true;
          for (size_t i = 0; i < expected.size(); ++i) {
            if (!std::isfinite(actual[i])) {
              finite = false;
              continue;
            }
            const float error = std::fabs(actual[i] - expected[i]);
            if (!std::isfinite(error)) {
              finite = false;
              continue;
            }
            if (error > max_error) {
              max_error = error;
            }
          }
          std::cout << "correct_output[0..5]=";
          for (size_t i = 0; i < expected.size(); ++i) {
            std::cout << (i == 0 ? "" : ",") << actual[i];
          }
          std::cout << "\nmax_abs_error=" << max_error << "\n";

          const bool ok = finite && max_error <= 1.0e-4F;
          DestroyDataset(&inputs);
          DestroyDataset(&outputs);
          cleanup();
          if (!ok) {
            std::cerr << "numerical check failed\n";
            return 1;
          }
          std::cout << "[OK] ACL descriptor/buffer 排障实战完成\n";
          return 0;
        }
        '''
    )
    source_path = workdir / "acl_buffer_troubleshooting.cpp"
    binary_path = workdir / "acl_buffer_troubleshooting"
    source_path.write_text(runner_source, encoding="utf-8")

    compile_cmd = [
        "g++",
        "-std=c++17",
        "-O2",
        "-D_GLIBCXX_USE_CXX11_ABI=0",
        "-Dgoogle=ascend_private",
        "-I" + str(prefix / "include"),
        str(source_path),
    ]
    for directory in library_dirs:
        compile_cmd.append("-L" + str(directory))
    compile_cmd.extend(
        [
            "-Wl,--no-as-needed",
            "-Wl,-rpath," + rpath,
            "-lacl_mdl",
            "-lacl_rt",
            "-lge_runner",
            "-lge_compiler",
            "-lgraph",
            "-lgraph_base",
            "-lge_common_base",
            "-o",
            str(binary_path),
        ]
    )
    compile_result = subprocess.run(
        compile_cmd, text=True, capture_output=True
    )
    if compile_result.returncode != 0:
        print(compile_result.stdout)
        print(compile_result.stderr)
        print("[SKIP] ACL 排障 runner 编译失败；请确认 CANN toolkit 的开发库已安装。")
        return
    print("[OK] ACL 排障 runner 编译完成：", binary_path)

    if om_path is None or not om_path.is_file():
        print(
            "[SKIP] 未找到 OM。请设置 GE_TROUBLESHOOTING_OM，"
            "例如 /tmp/ge_tf_parser_demo/tf_add.om。"
        )
        return
    runner_env = os.environ.copy()
    runner_env.setdefault("ASCEND_SLOG_PRINT_TO_STDOUT", "1")
    runner_env["LD_LIBRARY_PATH"] = (
        rpath + os.pathsep + runner_env.get("LD_LIBRARY_PATH", "")
    )
    result = subprocess.run(
        [str(binary_path), str(om_path)],
        text=True,
        capture_output=True,
        env=runner_env,
    )
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError("ACL/NPU 排障 runner 执行失败，请查看上方日志。")


run_acl_buffer_troubleshooting()


### 5.3 Device 内存与 stream 同步类



**内存 OOM** 定位思路（自上而下）：

```
1) 排查 C 原生接口：是否用 malloc/memset 申请 Host 内存 → 可用 asan 检测
2) 排查 CANN 内存申请：在 plog 搜 "mem_stats" 看各组件内存统计
   - aclrtMallocHost 报错 → Host 内存 OOM
   - aclrtMalloc / aclrtMallocPhysical 报错 → Device 内存 OOM
3) 其他报错 → Device 业务进程异常，搜 "DEV_PROC_MEM" 看各进程内存统计
```

```bash
grep "mem_stats" plog-*.log        # CANN 各组件内存统计
grep "DEV_PROC_MEM" plog-*.log     # Device 业务进程内存统计
```

**stream 同步 / 超时**：典型错误码 `EE1002 Stream_Synchronize_Timeout`。常见根因：算子执行超时（可能 AI Core Error 引发）、流上任务阻塞、Host/Device 协同异常。结合 AI Core Error 专题（msaicerr）与卡住/中断专题排查。

> 进程中断 / 进程卡住 也属运行时问题：卡住可用 **asys 实时堆栈导出**；中断/coredump 可用 asys 解析 coredump/stackcore，并查 plog 首报错。

## 6. 数据 dump 与精度定位

图结构没问题但**结果数值不对**时，需要 dump 算子的**输入/输出张量数据**（区别于第 3 节的「图结构 dump」）。

GE 数据 dump 的配置入口取决于 dump 类型，不能用一条固定的「优先级」概括。CANN 9.0 的普通数据/溢出 dump 主要通过 GE Option 配置；ACL 模型接口和异常 dump 还有各自独立的入口。

| 方式 | CANN 9.0 关键入口 | 说明 |
| --- | --- | --- |
| GE Option：普通数据 dump | `ge.exec.enableDump=1`、`ge.exec.dumpPath`、`ge.exec.dumpMode`、`ge.exec.dumpStep`、`ge.exec.dumpData`、`ge.exec.dumpLayer` | 通过 GE global options 或对应构图/编译接口传入；`dumpMode` 取 `input` / `output` / `all` |
| GE Option：溢出/调试 dump | `ge.exec.enableDumpDebug=1`（并配置 `ge.exec.dumpPath` 等） | 与 `ge.exec.enableDump=1` 不能同时开启，具体以版本文档约束为准 |
| ACL 模型接口 | `aclInit(...)` → `aclmdlInitDump()` → `aclmdlSetDump("/path/to/acl.json")` → 加载/执行模型 → `aclmdlFinalizeDump()` | 配置对接口调用后加载的模型生效；不是可任意切换的通用动态开关 |
| 异常 dump | `ge.exec.enable_exception_dump`；环境变量 `NPU_COLLECT_PATH` | 与普通数据 dump 通路不同；配置 `NPU_COLLECT_PATH` 时按其指定目录收集异常现场 |

dump 数据的三种场景：

| 场景 | 触发 | 内容 |
| --- | --- | --- |
| 普通数据 dump | 每次迭代执行 | 指定算子的 input/output 张量 |
| 溢出检测 dump | 检测到数值溢出（需 op debug 模式） | 溢出算子现场 |
| 异常 dump | 执行异常时 | 张量数据 + tiling_data / args / workspace 上下文 |

dump 文件名约定（便于定位）：

```
[场景]_[模型名]_[算子名]_[算子类型]_[迭代号]_[流ID]_[任务ID]
```

精度定位思路：

```
1) 开 dump（建议先 output、定位到可疑算子再 input/all）
2) 找到第一个「输出明显异常」的算子（NaN/Inf/数量级离谱）
3) 看它的输入是否已异常：是→继续往上游找；否→该算子自身问题
4) 离线工具解析 dump 数据（GE 不负责解析，由离线工具完成）
```

> 注意：L1/L1Fusion 上的张量不支持直接 dump；RT2.0（动态 shape）地址在执行后才确定，dump 在节点执行后触发。

## 7. 常用定位工具速查

| 工具 | 解决什么 | 典型用法 / 能力 |
| --- | --- | --- |
| **plog 日志** | 一切排障起点 | `grep ERROR/mem_stats/not supported` 看首报错与上下文 |
| **msnpureport** | 导出 Device 侧系统/内核态日志 | 把 Device 维测信息导到 Host 再分析 |
| **asys** | 一键式故障信息收集 + 多种解析（仅 Ascend EP 形态） | 收集软硬件/日志/dump；coredump/stackcore/trace 解析；**实时堆栈导出（卡住场景）**；AI Core Error 解析；健康检查 |
| **msaicerr** | 专攻 **AI Core Error** 分析（本地分析、依赖 python3.7.5+） | 分析 AI Core Error、解析 dump 文件、检查环境，输出分析报告 `info.txt` |
| **msprof** | 性能分析（profiling） | 采集 API/Host/Device 各层耗时，看算子级耗时、FP/BP、HCCL、AI Core metrics |
| **ascend-dmi** | 硬件压测 | `ascend-dmi --dg -i aicore -s` 压测 AI Core，辅助判断硬件故障 |
| **Netron** | 图可视化 | 打开 `ge_onnx*.pbtxt` 看图结构/连边/算子属性 |

### asys 收集（卡住场景导堆栈，示意）

```bash
# asys 仅支持 Ascend EP 形态；功能包括故障信息收集、复跑收集、健康检查、文件解析等
# 业务进程卡住时，可用「实时堆栈导出」功能导出堆栈定位
# AI Core Error 报错（如 "there is an aicore error exception"）可用 asys 的 AI Core Error 解析
```

### msaicerr 分析 AI Core Error（示意）

```bash
# 准备：收集 CANN 日志、exception dump、算子编译信息(*.o/*.json)
# 在「同一运行环境」本地执行 msaicerr 分析，得到 info.txt 分析报告
# 不支持分析部分通信类算子（如 MatmulAllReduce 系列、AllGatherMatmul 等）
```

### Profiling 开启（msprof，示意）

```bash
# CANN 9.0 环境变量方式（名称不带 GE_ 前缀）
export PROFILING_MODE=true
export PROFILING_OPTIONS='{"output":"/tmp/prof","training_trace":"on","task_trace":"on","aic_metrics":"PipeUtilization"}'
# 也可通过 GE Option：ge.exec.profilingMode=1 / ge.exec.profilingOptions=<JSON>
# 或 C API：aclgrphProfInit → aclgrphProfStart → aclgrphProfStop → aclgrphProfFinalize
```

> `PROFILING_MODE/PROFILING_OPTIONS` 环境变量主要用于 TensorFlow 训练和在线推理等场景；通用 GE 或 PyTorch 图编译场景，优先按当前接口使用 GE Option 或对应 Profiling API。工具可按问题类型选择：先用 plog 确认首个错误；编译和图结构问题使用 dump 图与 Netron；AI Core Error 使用 msaicerr；性能问题使用 msprof；信息不足时使用 asys 收集现场。

## 8. 排障总流程图

综合前面的定位方法，可以按以下顺序排查问题：

<p align="left"><img src="./images/troubleshooting_decision_tree.svg" alt="问题定位决策树" width="95%"></p>

排查时重点关注以下三点：

1. **先看错误码与首报错日志**（异步场景从首报错算子起）；
2. **编译问题 dump 图、运行问题核对 buffer/资源**；
3. **专项问题用专项工具**（AI Core Error→msaicerr、卡住→asys、性能→msprof）。

## 课后练习

本节介绍了 GE 用户的常见问题定位方法，请完成以下题目自测。

1. （判断题）`export ASCEND_GLOBAL_LOG_LEVEL=1` 表示把 CANN 日志级别设为 INFO，比默认更详细，常用于排障。

2. （判断题）错误码 `E19999`（形如 E*9***）一般表示用户类配置错误，按手册即可自助解决，无需联系技术支持。

3. （单选题）想看「经过 GE 优化、编译后的最终生成图」，应如何设置 dump？
    A. `DUMP_GE_GRAPH=3`
    B. `DUMP_GRAPH_LEVEL=3`
    C. `DUMP_GRAPH_FORMAT=readable`
    D. `ASCEND_GLOBAL_LOG_LEVEL=0`

4. （单选题）三种图 dump 格式中，「信息最完整、可转 JSON」的是哪种？
    A. readable（`ge_readable*.txt`）
    B. onnx（`ge_onnx*.pbtxt`）
    C. ge_proto（`ge_proto*.txt`）
    D. plog

5. （单选题）执行时报 `Output Size mismatch ... model expect ..., but given ...`，最合理的处理是？
    A. 降低日志级别
    B. 按报错提示的 Expected size 重新分配输入/输出 buffer，并核对输入 shape/dtype
    C. 重新安装 driver
    D. 关闭图 dump

6. （单选题）日志中出现 `IR for op[%s] optype[%s] is not registered.`，最可能的原因是？
    A. 输出 buffer 太小
    B. 算子原型 so 未加载成功，或原型未编译进 so（可用 nm -D 查符号表）
    C. stream 同步超时
    D. soc_version 不匹配

7. （多选题）以下工具与适用场景的对应关系，哪些正确？
    A. msaicerr：分析 AI Core Error 问题，输出 info.txt 报告
    B. asys：一键式故障信息收集，业务卡住时可实时堆栈导出
    C. msprof：性能分析，采集算子级耗时 / FP-BP / AI Core metrics
    D. Netron：打开 `ge_onnx*.pbtxt` 可视化查看图结构与算子属性

8. （多选题）内存 OOM 定位时，以下做法正确的有？
    A. 在 plog 中搜索 `mem_stats` 查看各组件内存统计
    B. `aclrtMallocHost` 报错通常指向 Host 内存 OOM
    C. `aclrtMalloc` / `aclrtMallocPhysical` 报错通常指向 Device 内存 OOM
    D. 搜索 `DEV_PROC_MEM` 可查看 Device 业务进程内存统计

9. （多选题）关于编译失败 InferShape 问题的定位，以下正确的有？
    A. 可 dump 含 `AfterInfershape` 的图，对比推导前后报错算子的 shape
    B. 检查报错算子各输入的 shape/dtype/format 是否满足算子原型约束
    C. 动态 shape 场景需核对 `INPUT_SHAPE` 范围/档位是否覆盖真实输入
    D. 顺数据流向上找「第一个 shape 不对」的算子，往往是真正源头

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/05.04_answer.txt